# LSTM CTC Baseline (Single-User)
This notebook implements a recurrent encoder baseline for the emg2qwerty single-user split.

Notes:
- This environment is CPU-only and `CTCLoss` may not be available on macOS.
- The cells below focus on model construction and a forward-pass sanity check.
- For full training, use a GPU runtime (e.g., Colab) and the CLI command at the end.

## Input/Output Shapes
After `LogSpectrogram` preprocessing, each input batch item has shape `(T, bands=2, C=16, freq=33)` where `freq = n_fft//2 + 1` for `n_fft=64`. The model expects a full batch shaped `(T, N, 2, 16, 33)` and returns emissions with shape `(T, N, num_classes)`.

## Config Reference
This notebook mirrors `config/model/lstm_ctc.yaml` for the recurrent encoder settings.

## Baseline Directions 2-5: Experiment Plan
1. Preprocessing/Augmentation: tune SpecAugment, band rotation offsets, temporal jitter, and log-spectrogram params.
2. Channels vs CER: keep tensor shape but zero out subsets of channels to emulate fewer electrodes.
3. Data amount vs CER: train on subsets of sessions (25/50/75/100%) and compare CER.
4. Sampling rate vs CER: downsample the raw EMG in time (stride) before spectrogram.

In [ ]:
import torch
from omegaconf import OmegaConf
from emg2qwerty.lightning import RecurrentCTCModule


In [ ]:
from emg2qwerty.transforms import (
    Compose,
    Lambda,
    LogSpectrogram,
    RandomBandRotation,
    SpecAugment,
    TemporalAlignmentJitter,
    ToTensor,
)

def mask_channels(keep_idx):
    keep_idx = torch.as_tensor(keep_idx)
    def _mask(x):
        # x shape: (T, bands, C)
        mask = torch.zeros(x.shape[-1], device=x.device)
        mask[keep_idx] = 1.0
        return x * mask
    return Lambda(_mask)

def temporal_downsample(stride):
    def _downsample(x):
        # x shape: (T, bands, C)
        return x[::stride]
    return Lambda(_downsample)

train_transform = Compose([
    ToTensor(),
    # mask_channels(list(range(8))),  # example: keep 8 channels
    # temporal_downsample(2),  # example: 2x downsample (2kHz -> 1kHz)
    RandomBandRotation(offsets=[-1, 0, 1]),
    TemporalAlignmentJitter(max_offset=120),
    LogSpectrogram(n_fft=64, hop_length=16),
    SpecAugment(n_time_masks=3, time_mask_param=25, n_freq_masks=2, freq_mask_param=4),
])

## Data Amount Subsets
Create smaller training sets by selecting fewer sessions in `config/user/single_user.yaml` or by creating new user configs (e.g., `single_user_25.yaml`). Track CER versus percent of data.

## Sampling Rate Notes
Downsampling with `temporal_downsample(stride)` reduces the effective sampling rate while keeping model input shape stable. You can also adjust `hop_length` in `LogSpectrogram` to change temporal resolution.

## Example Experiment Grid (CLI or Notebook)
1. Preprocessing: try `transforms.specaug.n_time_masks=0,2,4` and `transforms.logspec.n_fft=64,128`.
2. Channels: in notebooks, use `mask_channels(...)`; for CLI, consider a new transforms config that applies the same mask.
3. Data amount: define `config/user/single_user_25.yaml`, `single_user_50.yaml`, etc.
4. Sampling rate: use `temporal_downsample(2)` or increase `logspec.hop_length` to reduce time resolution.

In [ ]:
optimizer_cfg = OmegaConf.create({
    "_target_": "torch.optim.Adam",
    "lr": 1e-3,
})
lr_scheduler_cfg = OmegaConf.create({
    "scheduler": {
        "_target_": "torch.optim.lr_scheduler.CosineAnnealingLR",
        "T_max": 10,
    },
    "interval": "epoch",
})
decoder_cfg = OmegaConf.create({
    "_target_": "emg2qwerty.decoder.CTCGreedyDecoder"
})

model = RecurrentCTCModule(
    in_features=528,
    mlp_features=[384],
    rnn_type="lstm",
    hidden_size=256,
    num_layers=2,
    bidirectional=True,
    dropout=0.2,
    optimizer=optimizer_cfg,
    lr_scheduler=lr_scheduler_cfg,
    decoder=decoder_cfg,
)
model.eval()

In [ ]:
T, N = 200, 2
x = torch.randn(T, N, 2, 16, 33)
with torch.no_grad():
    y = model(x)
print('inputs:', x.shape)
print('emissions:', y.shape)

## Training (GPU Recommended)
Run the following from a GPU environment (e.g., Colab). Adjust batch size and epochs as needed.
```bash
python -m emg2qwerty.train user=single_user model=lstm_ctc \
  trainer.accelerator=gpu trainer.devices=1
```
If you only want a quick smoke test on CPU, you can set `trainer.accelerator=cpu` and `trainer.devices=1`, but CTC loss may not be available on macOS.